In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost xgboost tqdm -q

clear_output()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns     # Statistical visualization library, built on matblotlib
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q3_data.csv")

df = pd.read_csv(csv_path)
#df = df.drop(columns="Unnamed: 0", axis=1) # there was a column with no header so pandas names it unnamed, we need to drop this column when it happens


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
# Task 2: Write your code here:
# 2. Do we have missing values?

missing_values = df.isnull().sum()            # df.isnull() → Returns a DataFrame of the same shape with True where the value is NaN
print("Missing Values per Column:")
print(missing_values[missing_values > 0])     # Print missing columns only that has at least one missing value
if missing_values.any():                      # Check if any missing values exist overall
  print("\nHandle Missing Values as needed.")
else:
  print("\nNo Missing Values Found.")

print(missing_values)


for col in missing_values:
    df[col] = df[col].fillna(missing_values.mean())

In [ ]:
# Task 2: Write your code here:
# Task 3: Write your code here:
# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
# Task 4: Write your code here:
# 3. Do we have categorical columns?
categorical_cols = df.select_dtypes(include=["object"]).columns     # return which columns in my dataset are categorical (returns the columns names)

print("Categorical Columns:", categorical_cols)               # print it as list for cleaner output

#categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs', 'Delivery_Time', 'Vehicle_Type']

from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le

df

In [ ]:
# Task 4: Write your code here:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["number"]).columns.drop("Target")  # DON'T SCALE THE TARGET. include=["int64", "float64"] we can just put include=["number"]
print("Numerical Columns:", numerical_cols)

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()


In [ ]:
# Task 5: Write your code here:
# 1. Is the target imbalanced? i guess we only check if our data is impalanced in classification **make sure
def check_target_imbalance(df, target_column):
  print("Target Distribution (imbalanced):")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Target")

In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1).astype(float)
y = df['Target'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)


# Storage for logistic regression results for each fold
lr_losses = []
lr_accuracy = []
lr_precision = []
lr_recall = []
lr_f1 = []

# Define Model
model = CatBoostClassifier(
      verbose=0,            # verbose=0: Suppresses training output.
      n_estimators=320,
      max_depth=4
  )

# StratifiedKFold Cross Validation
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model.fit(X_train, y_train) # train
    y_pred = model.predict(X_test) # validate


    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Store results
    lr_accuracy.append(accuracy)
    lr_precision.append(precision)
    lr_recall.append(recall)
    lr_f1.append(f1)

    print("LOGISTIC REGRESSION Performance:")
    print(f"  Accuracy:  {np.mean(lr_accuracy):.4f}")
    print(f"  Precision: {np.mean(lr_precision):.4f}")
    print(f"  Recall:    {np.mean(lr_recall):.4f}")
    print(f"  F1-Score:  {np.mean(lr_f1):.4f}")


In [ ]:
# Task 1: Write your code here:
# Gather importances from the models (from the last fold)
importances = {}

importances = model.feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))    # Creates three horizontal subplots side by side (1 row, 3 columns).
axes = axes.flatten()
features = X.columns     # names of the input features for labeling.

for i, (model_name, imp) in enumerate(importances):
  # Sort features by importance for a cleaner plot, np.argsort(imp) → gives indices to sort features from least to most important.
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: